##  Section 1: Imports and configuration

In [1]:
import os
import gc
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import jax
import jax.numpy as jnp
from flax import linen as nn
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.facecolor"] = "white"

NUM_CLASSES = 4
PARTICLE_NAMES = ['Pion', 'Kaon', 'Proton', 'Electron']
COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

REAL_DATA_CSV = "/kaggle/input/rawreal-data-ao2d-lhc23zzh0544122/pid_features_real.csv"
MODEL_PATH = "/kaggle/input/trained_models/full_JAX_FSE_Attention.pkl"

TRAINING_FEATURES = [
    'pt', 'eta', 'phi',
    'tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el',
    'tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el',
    'bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el',
    'dca_xy', 'dca_z',
    'has_tpc', 'has_tof',
]

DETECTOR_GROUPS = {
    'tpc': ['tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el'],
    'tof': ['tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el'],
    'bayes': ['bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el'],
    'kinematics': ['pt', 'eta', 'phi', 'dca_xy', 'dca_z'],
}

## Section 2: FSE model

In [ ]:
class JAX_FSE_Attention(nn.Module):
    hidden_dim: int = 64
    num_heads: int = 4
    num_classes: int = 4
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, group_mask, training=False):
        batch_size = x.shape[0]
        num_groups = group_mask.shape[1]
        feat_proj = nn.Dense(self.hidden_dim * num_groups)(x)
        feat_proj = feat_proj.reshape(batch_size, num_groups, self.hidden_dim)
        feat_proj = feat_proj * group_mask[:, :, None]
        attn_mask = group_mask[:, None, None, :]
        feat_attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(
            feat_proj, feat_proj, mask=attn_mask
        )
        feat_attn = nn.LayerNorm()(feat_attn)
        gates = nn.sigmoid(nn.Dense(self.hidden_dim)(feat_attn))
        feat_gated = feat_attn * gates
        denom = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), 1.0)
        pooled = jnp.sum(feat_gated * group_mask[:, :, None], axis=1) / denom
        x = nn.Dense(128)(pooled)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(64)(x)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)

## Section 3: Load model

In [ ]:
with open(MODEL_PATH, "rb") as f:
    fse_results = pickle.load(f)

params = fse_results["params"]
hyper = fse_results.get("hyperparameters", {
    "hidden_dim": 128,
    "num_heads": 8,
    "dropout_rate": 0.4
})

model = JAX_FSE_Attention(
    hidden_dim=hyper["hidden_dim"],
    num_heads=hyper["num_heads"],
    num_classes=NUM_CLASSES,
    dropout_rate=hyper["dropout_rate"],
)

@jax.jit
def fse_forward(params, x, group_mask):
    return model.apply({"params": params}, x, group_mask, training=False)

## Section 4: Load real data

In [ ]:
print("Loading real data...")
df_real = pd.read_csv(
    REAL_DATA_CSV,
    na_values=['-', 'nan', 'null', 'NaN', 'NULL', ''],
    keep_default_na=True,
    on_bad_lines='skip'
)
print("Shape:", df_real.shape)

## Section 5: Preprocess for inference

In [ ]:
df_proc = df_real.copy()
bayes_features = DETECTOR_GROUPS["bayes"]

for feat in DETECTOR_GROUPS["tof"]:
    if feat in df_proc.columns:
        df_proc[feat] = df_proc[feat].fillna(0.0 if feat == "tof_beta" else 999.0)

for feat in DETECTOR_GROUPS["tpc"]:
    if feat in df_proc.columns:
        df_proc[feat] = df_proc[feat].fillna(0.0 if feat == "tpc_signal" else 999.0)

for feat in bayes_features:
    if feat in df_proc.columns:
        miss = (df_proc[feat] == 0) | (df_proc[feat].isna())
        df_proc[f"{feat}_missing"] = miss.astype("float32")
        df_proc.loc[miss, feat] = -0.25
        miss_col = f"{feat}_missing"
        if miss_col not in TRAINING_FEATURES:
            TRAINING_FEATURES.append(miss_col)

for feat in DETECTOR_GROUPS["kinematics"]:
    if feat in df_proc.columns:
        df_proc[feat] = df_proc[feat].fillna(df_proc[feat].median())

for col in ["has_tpc", "has_tof"]:
    if col in df_proc.columns:
        df_proc[col] = df_proc[col].fillna(0.0)

used_features = [f for f in TRAINING_FEATURES if f in df_proc.columns]
X_real = df_proc[used_features].values.astype("float32")

scaler = StandardScaler()
X_scaled = X_real.copy()

for i, feat in enumerate(used_features):
    if feat.endswith("_missing"):
        continue
    col = X_real[:, i]
    if feat in bayes_features:
        real_mask = col != -0.25
        if real_mask.sum() > 0:
            scaler.fit(col[real_mask].reshape(-1, 1))
            X_scaled[real_mask, i] = scaler.transform(col[real_mask].reshape(-1, 1)).reshape(-1)
    else:
        scaler.fit(col.reshape(-1, 1))
        X_scaled[:, i] = scaler.transform(col.reshape(-1, 1)).reshape(-1)

group_masks = np.stack([
    df_proc["has_tpc"].values.astype("float32"),
    df_proc["has_tof"].values.astype("float32"),
    np.ones(len(df_proc), dtype="float32"),
], axis=1)

print("Feature matrix:", X_scaled.shape)
print("Group mask:", group_masks.shape)

## Section 6: Inference

In [ ]:
batch_size = 4096
n = len(df_proc)
y_pred = np.empty(n, dtype=np.int32)
y_proba = np.empty((n, NUM_CLASSES), dtype=np.float32)
confidence = np.empty(n, dtype=np.float32)

print("Running inference...")
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    logits = fse_forward(
        params,
        jnp.array(X_scaled[start:end]),
        jnp.array(group_masks[start:end])
    )
    probs = np.array(jax.nn.softmax(logits, axis=-1))
    y_proba[start:end] = probs
    y_pred[start:end] = probs.argmax(axis=1)
    confidence[start:end] = probs.max(axis=1)

df_proc["ml_pred_class"] = y_pred
df_proc["ml_prob_pi"] = y_proba[:, 0]
df_proc["ml_prob_ka"] = y_proba[:, 1]
df_proc["ml_prob_pr"] = y_proba[:, 2]
df_proc["ml_prob_el"] = y_proba[:, 3]
df_proc["ml_confidence"] = confidence

print("Inference done.")
print("Mean confidence:", df_proc["ml_confidence"].mean())

## Section 7: Save output

In [ ]:
OUTPUT_CSV = "ml_predictions_real_data_FSE.csv"
df_proc.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

## Section 8: Basic validation

In [ ]:
MOMENTUM_RANGES = {
    'mr_0_1': {'name': '0–1 GeV/c', 'min': 0.0, 'max': 1.0},
    'mr_1_3': {'name': '1–3 GeV/c', 'min': 1.0, 'max': 3.0},
    'mr_full': {'name': 'Full Spectrum', 'min': 0.0, 'max': 100.0},
}

print("\n" + "="*80)
print("PHYSICS VALIDATION")
print("="*80)

print("\nBasic prediction fractions:")
for i, name in enumerate(PARTICLE_NAMES):
    frac = (df_proc["ml_pred_class"] == i).mean() * 100
    print(f"  {name:<8s}: {frac:6.2f}%")

print("\nConfidence statistics:")
print(f"  Mean   : {df_proc['ml_confidence'].mean():.4f}")
print(f"  Median : {df_proc['ml_confidence'].median():.4f}")
print(f"  >0.90  : {(df_proc['ml_confidence'] > 0.90).mean() * 100:.1f}%")
print(f"  >0.95  : {(df_proc['ml_confidence'] > 0.95).mean() * 100:.1f}%")

print("\nTPC dE/dx ordering check:")
for i, name in enumerate(PARTICLE_NAMES):
    mask = (df_proc["ml_pred_class"] == i) & (df_proc["tpc_signal"] > 0)
    if mask.sum() > 0:
        print(f"  {name:<8s}: {df_proc.loc[mask, 'tpc_signal'].mean():.3f}")

print("\nTOF beta ordering check:")
for i, name in enumerate(PARTICLE_NAMES):
    mask = (df_proc["ml_pred_class"] == i) & (df_proc["tof_beta"] > 0) & (df_proc["tof_beta"] < 2)
    if mask.sum() > 0:
        print(f"  {name:<8s}: {df_proc.loc[mask, 'tof_beta'].mean():.4f}")

## Section 9: pT bin validation

In [ ]:
pt_bins = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 100.0]
pt_labels = ["0-0.5", "0.5-1.0", "1.0-1.5", "1.5-2.0", "2.0-3.0", "3.0-5.0", "5.0-10.0", ">10"]
df_proc["pt_bin"] = pd.cut(df_proc["pt"], bins=pt_bins, labels=pt_labels, include_lowest=True)

print("\nFractions vs pT:")
for label in pt_labels:
    sub = df_proc[df_proc["pt_bin"] == label]
    if len(sub) == 0:
        continue
    print(f"\n{label}  (N={len(sub):,})")
    for i, name in enumerate(PARTICLE_NAMES):
        frac = (sub["ml_pred_class"] == i).mean() * 100
        print(f"  {name:<8s}: {frac:6.2f}%")

## Section 10: Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fractions = [(df_proc["ml_pred_class"] == i).mean() * 100 for i in range(4)]
ax.bar(PARTICLE_NAMES, fractions, color=COLORS, edgecolor="black")
ax.set_ylabel("Fraction (%)")
ax.set_title("ML Predicted Particle Fractions")
plt.tight_layout()
plt.savefig("plot_predicted_fractions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
rows = []
for label in pt_labels:
    sub = df_proc[df_proc["pt_bin"] == label]
    if len(sub) == 0:
        continue
    rows.append({
        "pt_bin": label,
        "Pion": (sub["ml_pred_class"] == 0).mean() * 100,
        "Kaon": (sub["ml_pred_class"] == 1).mean() * 100,
        "Proton": (sub["ml_pred_class"] == 2).mean() * 100,
        "Electron": (sub["ml_pred_class"] == 3).mean() * 100,
    })

plot_frac = pd.DataFrame(rows)
plot_frac.set_index("pt_bin")[PARTICLE_NAMES].plot(kind="bar", stacked=True, ax=ax, color=COLORS)
ax.set_ylabel("Fraction (%)")
ax.set_title("ML Particle Fractions vs pT")
plt.tight_layout()
plt.savefig("plot_fractions_vs_pt.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_proc["ml_confidence"], bins=50, color="steelblue", edgecolor="black")
ax.set_xlabel("ML confidence")
ax.set_ylabel("Tracks")
ax.set_title("Confidence Distribution")
plt.tight_layout()
plt.savefig("plot_confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()